# Linear discriminant analysis

Use LDA as a supervised one-dimensional projection and compare it with a pass/fail classifier.

**Dataset:** `data/student_data.csv` (395 student records, 33 columns). **Target:** final grade `G3` (0–20). For classification labs, pass means `G3 ≥ 10`. Earlier grades `G1` and `G2` are included as predictors; interpret results as prediction after those earlier grades are known. The split is fixed with random seed 42 for reproducibility.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [Path("data/student_data.csv"), Path("../data/student_data.csv")]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place student_data.csv in the repository's data/ folder.")
df = pd.read_csv(DATA_PATH)
TARGET = "G3"  # final-period grade (0–20)
PASS_THRESHOLD = 10
numeric_features = df.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y_reg = df[TARGET]
y_class = (df[TARGET] >= PASS_THRESHOLD).astype(int)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def make_preprocessor(scale_numeric=True):
    numeric_steps = [("scale", StandardScaler())] if scale_numeric else []
    numeric_pipe = Pipeline(numeric_steps) if numeric_steps else "passthrough"
    return ColumnTransformer(
        [("numeric", numeric_pipe, numeric_features),
         ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
        remainder="drop",
    )


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report

X_train,X_test,y_train,y_test=train_test_split(X,y_class,test_size=0.25,stratify=y_class,random_state=42)
prep=make_preprocessor(); A=prep.fit_transform(X_train); B=prep.transform(X_test)
if hasattr(A, "toarray"):
    A, B = A.toarray(), B.toarray()
lda=LinearDiscriminantAnalysis(n_components=1)
Z_train=lda.fit_transform(A,y_train).ravel()
Z_test=lda.transform(B).ravel()
print("LDA explained variance ratio:",np.round(lda.explained_variance_ratio_,3))
plt.figure(figsize=(8,4))
for label,name in [(0,"Below 10"),(1,"Pass")]: sns.kdeplot(Z_train[y_train.to_numpy()==label],label=name,fill=True,alpha=.25)
plt.xlabel("LDA component 1"); plt.title("Training-set class separation in LDA space"); plt.legend(); plt.show()
pred=lda.predict(B)
print("Held-out accuracy:",round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,target_names=["Below threshold","Pass"],zero_division=0))
